# Prática — Módulo 4 – Representação Vetorial Baseada em Coocorrência

##Aluno: Peterson Alves Pereira - CDN 5ºsem

## Prática
Nesta prática o aluno constrói passo a passo diferentes representações vetoriais a partir de um pequeno corpus textual. Inicialmente são exploradas estatísticas básicas do corpus, seguida da geração de matrizes BoW e TF-IDF. Em seguida são extraídos bigramas sequenciais para construir uma matriz de coocorrência palavra-contexto. A partir dessa matriz são calculadas probabilidades conjuntas, probabilidades marginais e, por fim, os valores de PMI e PPMI, permitindo observar como relações estatísticas entre palavras emergem diretamente dos dados.

## Dataset
O corpus utilizado consiste em um conjunto pequeno de frases sintéticas envolvendo agentes (como gato, cachorro e homem), verbos de ação (bebe, come) e alimentos ou líquidos (leite, água, carne, peixe). Embora simples, esse corpus é suficiente para ilustrar como padrões de coocorrência podem revelar associações entre palavras, como verbos e objetos, permitindo visualizar o funcionamento das matrizes de coocorrência e das medidas de associação utilizadas em PLN.




In [ ]:
# Importar bibliotecas
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [ ]:
# Definir um pequeno corpus para analisar coocorrência entre palavras.

# Corpus minúsculo.
corpus = [
    "gato bebe agua",
    "cachorro bebe agua",
    "gato bebe leite",
    "cachorro bebe leite",
    "gato come peixe",
    "cachorro come peixe",
    "gato come carne",
    "cachorro come carne",
    "homem bebe agua",
    "homem come carne",
]

corpus

# Visualizar o corpus.
for i, frase in enumerate(corpus, 1):
    print(f"D{i}: {frase}")

D1: gato bebe agua
D2: cachorro bebe agua
D3: gato bebe leite
D4: cachorro bebe leite
D5: gato come peixe
D6: cachorro come peixe
D7: gato come carne
D8: cachorro come carne
D9: homem bebe agua
D10: homem come carne


## PARTE 1 — BoW

### Pergunta
1. Quantas palavras diferentes existem no corpus?

R: O corpus possui 9 palavras diferentes em sua totalidade, sendo elas: bebe, come,gato, cachorro, agua, carne, leite, peixe e homem.

2. Quais palavras aparecem em mais de um documento?

R: Todas as palavras aparecem em mais de um dataset, sendo a frequência mínima de 2 vezes nas palavras: leite, peixe e homem.

In [ ]:
# Contar palavras no corpus.
from collections import Counter

tokens = " ".join(corpus).split()

freq = Counter(tokens)
freq.most_common()


[('bebe', 5),
 ('come', 5),
 ('gato', 4),
 ('cachorro', 4),
 ('agua', 3),
 ('carne', 3),
 ('leite', 2),
 ('peixe', 2),
 ('homem', 2)]

In [ ]:
# Criar matriz Bag of Words.
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(corpus)

bow = pd.DataFrame(
    X_bow.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"D{i+1}" for i in range(len(corpus))]
)

bow

,agua,bebe,cachorro,carne,come,gato,homem,leite,peixe
D1,1,1,0,0,0,1,0,0,0
D2,1,1,1,0,0,0,0,0,0
D3,0,1,0,0,0,1,0,1,0
D4,0,1,1,0,0,0,0,1,0
D5,0,0,0,0,1,1,0,0,1
D6,0,0,1,0,1,0,0,0,1
D7,0,0,0,1,1,1,0,0,0
D8,0,0,1,1,1,0,0,0,0
D9,1,1,0,0,0,0,1,0,0
D10,0,0,0,1,1,0,1,0,0


### Pergunta
3. Quantos vetores existem na matriz BoW?

R: Existem 10 vetores nessa matriz.

4. Cada vetor representa o quê?

R: Cada vetor (linha da matriz) representa um documento (ou frase) do corpus. O vetor é a representação numérica de todo o conteúdo daquela frase específica.

5. Qual é a dimensão de cada vetor?

R: Cada vetor possui 9 dimensões. É o número de colunas.

6. A dimensão do vetor depende de quê?

R: A dimensão do vetor depende do tamanho do vocabulário único do corpus. Não a quantidade total de palavras ou documentos.

7. Documentos que compartilham palavras terão vetores parecidos?

R: Sim, compartilham. Documentos que possuem as mesmas palavras terão valores 1 nas mesmas dimensões (colunas) da matriz. Quanto mais palavras em comum, mais próximas serão as coordenadas dos vetores no espaço vetorial, tornando-os matematicamente similares.

8. Pela matriz BoW, é possível perceber que “bebe” aparece tanto com “gato” quanto com “cachorro”? Ou essa relação não fica explícita na matriz?

R: Essa relação não fica explícita pois o BoW é um modelo que ignora a ordem e a estrutura das frases.

PARTE 2 — TF-IDF

### Pergunta
9. A matriz TF-IDF possui o mesmo número de vetores do BoW?

R: A matriz permanece com o mesmo número, 10 vetores.

10. A dimensão dos vetores mudou em relação ao BoW?

R: Não, a dimensão permanece como 9 em sua totalidade.

11. Por que agora aparecem valores decimais em vez de contagens inteiras?

R: O TF-IDF é uma técnica estatística que avalia a importância de uma palavra em um documento dentro de uma coleção. Os valores decimais aparecem pois ele faz uma ponderação estatística, ele atribui um "peso" para representar a "importância" de cada vetor.

12. Todas as palavras possuem o mesmo peso dentro de um documento?

R: Não, pois o TF-IDF não levar em conta apenas a frequência do termo no documento, ele também leva em conta a infrequência. Sendo assim, se uma palavra aparece em vários documentos, elas recebem peso menor e as que aparecem em menos documentos recebem um peso maior, ou seja, mais discrimativos.

13. Qual é a palavra mais informativa (mais discriminativa) de cada documento?

R: As palavras mais informativas são:



```
D1 e D2: agua (peso: 0.6418)
D3 e D4: leite (peso: 0.6912)
D5 e D6: peixe (peso: 0.6912)
D7 e D8: carne (peso: 0.6418)
D9 e D10: homem (peso: 0.6662)
```

In [ ]:
# gerar matriz TF-IDF.
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(corpus)

tfidf = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"D{i+1}" for i in range(len(corpus))]
)

tfidf

,agua,bebe,cachorro,carne,come,gato,homem,leite,peixe
D1,0.641771,0.512414,0.000000,0.000000,0.000000,0.570581,0.000000,0.000000,0.000000
D2,0.641771,0.512414,0.570581,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
D3,0.000000,0.482845,0.000000,0.000000,0.000000,0.537655,0.000000,0.691222,0.000000
D4,0.000000,0.482845,0.537655,0.000000,0.000000,0.000000,0.000000,0.691222,0.000000
D5,0.000000,0.000000,0.000000,0.000000,0.482845,0.537655,0.000000,0.000000,0.691222
D6,0.000000,0.000000,0.537655,0.000000,0.482845,0.000000,0.000000,0.000000,0.691222
D7,0.000000,0.000000,0.000000,0.641771,0.512414,0.570581,0.000000,0.000000,0.000000
D8,0.000000,0.000000,0.570581,0.641771,0.512414,0.000000,0.000000,0.000000,0.000000
D9,0.582818,0.465343,0.000000,0.000000,0.000000,0.000000,0.666168,0.000000,0.000000
D10,0.000000,0.000000,0.000000,0.582818,0.465343,0.000000,0.666168,0.000000,0.000000


### Pergunta
14. O TF-IDF consegue capturar relação entre palavras como bebe → leite?

R: Não, o TF-IDF identifica termos que estão no mesmo documento, mas ele não sabe se bebe está próximo a leite. Ele avalia o grau de importância, não o de proximidade.

In [ ]:
most_informative_tfidf = {}
for index, row in tfidf.iterrows():
    # Get the word with the maximum TF-IDF score in the current document
    max_word_tfidf = row.idxmax()
    max_tfidf_value = row.max()
    most_informative_tfidf[index] = (max_word_tfidf, max_tfidf_value)

print("Palavra mais informativa por documento (TF-IDF):")
for doc, (word, value) in most_informative_tfidf.items():
    print(f"{doc}: {word} (peso: {value:.4f})")

Palavra mais informativa por documento (TF-IDF):
D1: agua (peso: 0.6418)
D2: agua (peso: 0.6418)
D3: leite (peso: 0.6912)
D4: leite (peso: 0.6912)
D5: peixe (peso: 0.6912)
D6: peixe (peso: 0.6912)
D7: carne (peso: 0.6418)
D8: carne (peso: 0.6418)
D9: homem (peso: 0.6662)
D10: homem (peso: 0.6662)


## PARTE 3 — Coocorrência (n-gramas)

In [ ]:
# Gerar bigramas sequenciais (janela de contexto = "1 à direita").
from nltk.util import bigrams

bigrams_list = []

for sent in corpus:
    tokens = sent.split()
    bigrams_list.extend(list(bigrams(tokens)))

bigrams_list

[('gato', 'bebe'),
 ('bebe', 'agua'),
 ('cachorro', 'bebe'),
 ('bebe', 'agua'),
 ('gato', 'bebe'),
 ('bebe', 'leite'),
 ('cachorro', 'bebe'),
 ('bebe', 'leite'),
 ('gato', 'come'),
 ('come', 'peixe'),
 ('cachorro', 'come'),
 ('come', 'peixe'),
 ('gato', 'come'),
 ('come', 'carne'),
 ('cachorro', 'come'),
 ('come', 'carne'),
 ('homem', 'bebe'),
 ('bebe', 'agua'),
 ('homem', 'come'),
 ('come', 'carne')]

In [ ]:
# Contar frequência dos bigramas.
from collections import Counter

bigram_freq = Counter(bigrams_list)
bigram_freq

Counter({('gato', 'bebe'): 2,
         ('bebe', 'agua'): 3,
         ('cachorro', 'bebe'): 2,
         ('bebe', 'leite'): 2,
         ('gato', 'come'): 2,
         ('come', 'peixe'): 2,
         ('cachorro', 'come'): 2,
         ('come', 'carne'): 3,
         ('homem', 'bebe'): 1,
         ('homem', 'come'): 1})

In [ ]:
# Criar matriz de coocorrência a partir dos bigramas.
words = sorted(set([w for bg in bigram_freq for w in bg]))

matrix = pd.DataFrame(0, index=words, columns=words)

for (w,c),freq in bigram_freq.items():
    matrix.loc[w,c] = freq

matrix

,agua,bebe,cachorro,carne,come,gato,homem,leite,peixe
agua,0,0,0,0,0,0,0,0,0
bebe,3,0,0,0,0,0,0,2,0
cachorro,0,2,0,0,2,0,0,0,0
carne,0,0,0,0,0,0,0,0,0
come,0,0,0,3,0,0,0,0,2
gato,0,2,0,0,2,0,0,0,0
homem,0,1,0,0,1,0,0,0,0
leite,0,0,0,0,0,0,0,0,0
peixe,0,0,0,0,0,0,0,0,0


## PARTE 4 — PPMI

### Pergunta
15. O que significa a probabilidade conjunta P(w,c) para um par de palavras?

R: A probabilidade conjunta é a probabilidade da palavra, no caso (W) e seu contexto (c) aparecerem junto no corpo em relação ao total de pares possíveis, já que nesse caso estamos trabalhando com bigramas. É calculado dividindo a frequência de coocorrência do par pelo número total de bigramas no texto.

In [ ]:
# Calcular total de coocorrências.
N = matrix.values.sum()
N

np.int64(20)

In [ ]:
# Calcular probabilidade conjunta dos bigramas.
N = sum(bigram_freq.values())

p_wc = {bg: freq / N for bg, freq in bigram_freq.items()}
p_wc

{('gato', 'bebe'): 0.1,
 ('bebe', 'agua'): 0.15,
 ('cachorro', 'bebe'): 0.1,
 ('bebe', 'leite'): 0.1,
 ('gato', 'come'): 0.1,
 ('come', 'peixe'): 0.1,
 ('cachorro', 'come'): 0.1,
 ('come', 'carne'): 0.15,
 ('homem', 'bebe'): 0.05,
 ('homem', 'come'): 0.05}

### Pergunta
16. Qual é o valor da soma das probabilidades conjuntas? O resultado seria o mesmo independentemente da quantidade de palavras?

R: O valor da soma é sempre 1 (ou 100%). Sim, o resultado será o mesmo independentemente da quantidade de palavras no corpus, pois a soma de todas as probabilidades de um espaço amostral completo sempre resulta na unidade (a probabilidade total). A comprovação se dá logo na célula abaixo:

In [ ]:
# Gerar matrix de probabilidade conjunta.
N = matrix.values.sum()

p_wc = matrix / N
print(p_wc)
print(f"\nSoma total das probabilidades conjuntas: {p_wc.values.sum():.2f}")

          agua  bebe  cachorro  carne  come  gato  homem  leite  peixe
agua      0.00  0.00       0.0   0.00  0.00   0.0    0.0    0.0    0.0
bebe      0.15  0.00       0.0   0.00  0.00   0.0    0.0    0.1    0.0
cachorro  0.00  0.10       0.0   0.00  0.10   0.0    0.0    0.0    0.0
carne     0.00  0.00       0.0   0.00  0.00   0.0    0.0    0.0    0.0
come      0.00  0.00       0.0   0.15  0.00   0.0    0.0    0.0    0.1
gato      0.00  0.10       0.0   0.00  0.10   0.0    0.0    0.0    0.0
homem     0.00  0.05       0.0   0.00  0.05   0.0    0.0    0.0    0.0
leite     0.00  0.00       0.0   0.00  0.00   0.0    0.0    0.0    0.0
peixe     0.00  0.00       0.0   0.00  0.00   0.0    0.0    0.0    0.0

Soma total das probabilidades conjuntas: 1.00


### Pergunta
17. Se P(w,c) é alto, por si só, isso implica forte associação entre w e c?

R: Não necessariamente. Uma probabilidade conjunta alta pode ser apenas o resultado de uma (ou ambas) as palavras serem muito comuns no corpus. Por exemplo, palavras como "o" ou "de" aparecem com frequência ao lado de quase todas as outras palavras, sem que haja uma associação semântica forte ou específica entre elas.

18. Como saber se duas palavras aparecem juntas mais do que seria esperado ao acaso?

R: : A gente compara a chance de elas estarem juntas no texto com a chance de elas estarem separadas. Se a chance de estarem juntas for muito maior do que a chance de estarem separadas, a gente sabe que existe uma conexão real entre elas (como "bebe" e "agua") e não foi apenas por sorte ou coincidência.



In [ ]:
# Calcular frequência marginal das palavras.
p_w = matrix.sum(axis=1) / N
p_w

,0
agua,0.00
bebe,0.25
cachorro,0.20
carne,0.00
come,0.25
gato,0.20
homem,0.10
leite,0.00
peixe,0.00


In [ ]:
# Calcular frequência marginal dos contextos.
p_c = matrix.sum(axis=0) / N
p_c

,0
agua,0.15
bebe,0.25
cachorro,0.00
carne,0.15
come,0.25
gato,0.00
homem,0.00
leite,0.10
peixe,0.10


In [ ]:
# Calcular PMI (Pointwise Mutual Information).
import numpy as np

pmi = np.log(p_wc / (p_w.values[:, None] * p_c.values[None, :]))
pmi = pmi.replace([np.inf, -np.inf], 0)
pmi

/usr/local/lib/python3.12/dist-packages/pandas/core/internals/blocks.py:393: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)


,agua,bebe,cachorro,carne,come,gato,homem,leite,peixe
agua,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bebe,1.386294,0.000000,NaN,0.000000,0.000000,NaN,NaN,1.386294,0.000000
cachorro,0.000000,0.693147,NaN,0.000000,0.693147,NaN,NaN,0.000000,0.000000
carne,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
come,0.000000,0.000000,NaN,1.386294,0.000000,NaN,NaN,0.000000,1.386294
gato,0.000000,0.693147,NaN,0.000000,0.693147,NaN,NaN,0.000000,0.000000
homem,0.000000,0.693147,NaN,0.000000,0.693147,NaN,NaN,0.000000,0.000000
leite,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
peixe,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Pergunta
19. Porque apareceu NaN?

R: Porque o código roda um logarítimo, onde o valor é zero e ele divide por zero, computacionalmente aquele valor fica indefinido. Nesse caso a linguagem python preenche com um NaN (not a number).

20. O que acontece no passo chamado Positive PMI?

R: Ele funciona como um filtro na matriz, se o valor for positivo ele mantém e se for negativo, zero ou NaN ele transforma tudo em zero. Isso evita ruídos gerados por palavras que quase nunca aparecem juntas.

In [ ]:
# Converter para PPMI.
ppmi = pmi.clip(lower=0).fillna(0)
ppmi

,agua,bebe,cachorro,carne,come,gato,homem,leite,peixe
agua,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
bebe,1.386294,0.000000,0.0,0.000000,0.000000,0.0,0.0,1.386294,0.000000
cachorro,0.000000,0.693147,0.0,0.000000,0.693147,0.0,0.0,0.000000,0.000000
carne,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
come,0.000000,0.000000,0.0,1.386294,0.000000,0.0,0.0,0.000000,1.386294
gato,0.000000,0.693147,0.0,0.000000,0.693147,0.0,0.0,0.000000,0.000000
homem,0.000000,0.693147,0.0,0.000000,0.693147,0.0,0.0,0.000000,0.000000
leite,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
peixe,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000


### Pergunta
21. A matriz PPMI indica claramente quais alimentos estão associados a bebe e quais estão associados a come. No entanto, é possível determinar a partir dela qual agente (gato, cachorro, homem) bebe ou come cada alimento específico?

R: Não exatamente. A PPMI é boa pra mostrar que "bebe" e "leite" andam juntos, ou que "gato" e "bebe" aparecem um do lado do outro. Mas como o modelo só olha os pares (bigramas), ele não consegue "ligar os pontos" da frase inteira. Ele sabe que o gato faz a ação e o leite sofre a ação, mas a conexão direta entre o sujeito e o objeto se perde porque eles não são vizinhos de parede na frase.

22. Esse tipo de identificação de associação entre verbos e objetos (por exemplo, bebe–água ou come–peixe) era possível utilizando apenas BoW ou TF-IDF? Explique por quê.

Não, O BoW só conta palavra, então ele sabe que "bebe" tá lá, mas não sabe com quem. O TF-IDF é até pior pra isso, porque como "bebe" e "come" aparecem em quase todas as frases, o peso deles fica lá embaixo (são penalizados), aí fica impossível ver que eles têm uma relação forte com a "água" ou com o "peixe". Essa "química" entre o verbo e o objeto só aparece mesmo quando a gente usa o PMI pra ver quem coocorre mais do que o esperado.
